uv sync --extra notebooks

In [2]:
import logging
import geopandas as gpd
import os
from glob import glob
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def transform_projections(raw, transformed, target_epsg=5514):
    """
    Reads all geojson files in the raw data folder, transforms them to target_epsg,
    and saves them to the transformed folder.
    """
    # 1. Create output folder if it doesn't exist
    if not os.path.exists(transformed):
        os.makedirs(transformed)
        logger.info(f"Created output directory: {transformed}")

    # 2. Get a list of all .geojson files in the input folder
    search_path = os.path.join(raw, "*.geojson")
    files = glob(search_path)
    
    if not files:
        logger.warning("No .geojson files found in the specified folder.")
        return

    # 3. Loop through each file
    for file_path in files:
        filename = os.path.basename(file_path)
        try:
            logger.info(f"---- processing {filename} ----")
            
            # Load the file
            gdf = gpd.read_file(file_path)
            
            # Transform the projection
            logger.info(f"Adjusting projection to EPSG:{target_epsg}...")
            gdf = gdf.to_crs(epsg=target_epsg)
            
            # Define output path
            output_path = os.path.join(transformed, filename)
            
            # Save the file
            gdf.to_file(output_path, driver='GeoJSON')
            logger.info(f"Successfully saved to {output_path}")
            
        except Exception as e:
            logger.error(f"Failed to process {filename}: {e}")

# --- Execution ---
if __name__ == "__main__":
    #navigating to the data folder
    project_root = Path(r"C:\Users\kana2\OneDrive\Documents\GitHub\spatial-justice\data") 
    #current_file_path = Path(__file__).resolve()
    #project_root = current_file_path.parents[3]
    
    INPUT_DIR = project_root / "raw" 
    OUTPUT_DIR = project_root / "transformed"
    TARGET_EPSG = 5514

    transform_projections(str(INPUT_DIR), str(OUTPUT_DIR), TARGET_EPSG)

INFO:__main__:---- processing admin_boundaries_ORP.geojson ----
INFO:__main__:Adjusting projection to EPSG:5514...
INFO:pyogrio._io:Created 206 records
INFO:__main__:Successfully saved to C:\Users\kana2\OneDrive\Documents\GitHub\spatial-justice\data\transformed\admin_boundaries_ORP.geojson
INFO:__main__:---- processing admin_boundaries_ORP_points.geojson ----
INFO:__main__:Adjusting projection to EPSG:5514...
INFO:pyogrio._io:Created 206 records
INFO:__main__:Successfully saved to C:\Users\kana2\OneDrive\Documents\GitHub\spatial-justice\data\transformed\admin_boundaries_ORP_points.geojson
INFO:__main__:---- processing census_ORP.geojson ----
INFO:__main__:Adjusting projection to EPSG:5514...
INFO:pyogrio._io:Created 206 records
INFO:__main__:Successfully saved to C:\Users\kana2\OneDrive\Documents\GitHub\spatial-justice\data\transformed\census_ORP.geojson
INFO:__main__:---- processing OD_emergency_care.geojson ----
INFO:__main__:Adjusting projection to EPSG:5514...
INFO:pyogrio._io:Crea

original standardizing code

In [ ]:
from pathlib import Path
import geopandas as gpd

# Project root directory
ROOT = Path(__file__).resolve().parents[3]

input_file = ROOT / "data" / "transformed" / "admin_boundaries_ORP.geojson"
output_file = ROOT / "data" / "transformed" / "admin_boundaries_ORP_variables.geojson"

gdf = gpd.read_file(input_file)

print(input_file)

# CREATE STANDARDIZED VARIABLES

# Area in km²
gdf["AREA_KM2"] = gdf["SHAPE_Area"] / 1_000_000

# Population density
gdf["POP_DENS"] = (
    gdf["POCET_OBYV"] /
    gdf["AREA_KM2"]
)

# Percentage women
gdf["PCT_WOMEN"] = (
    gdf["ZENY"] /
    gdf["POCET_OBYV"]
) * 100

# Percentage children
gdf["PCT_CHILD"] = (
    gdf["OBYV_0_14"] /
    gdf["POCET_OBYV"]
) * 100

# Percentage working-age
gdf["PCT_WORKING"] = (
    gdf["OBYV_15_64"] /
    gdf["POCET_OBYV"]
) * 100

# Percentage elderly
gdf["PCT_ELDERLY"] = (
    gdf["OBYV_65"] /
    gdf["POCET_OBYV"]
) * 100

# Ageing index
gdf["AGEING_INDEX"] = (
    gdf["OBYV_65"] /
    gdf["OBYV_0_14"]
) * 100

# Dependency ratio
gdf["DEPENDENCY"] = (
    (gdf["OBYV_0_14"] + gdf["OBYV_65"])
    /
    gdf["OBYV_15_64"]
) * 100

# Natural increase
gdf["NATURAL_INC"] = (
    gdf["NAROZENI"] -
    gdf["ZEMRELI"]
)

# Natural increase rate per 1000 inhabitants
gdf["NATURAL_INC_RATE"] = (
    gdf["NATURAL_INC"]
    /
    gdf["POCET_OBYV"]
) * 1000

# Migration balance
gdf["MIG_BAL"] = (
    gdf["PRISTEHOVALI"] -
    gdf["VYSTEHOVALI"]
)

# Migration balance rate per 1000 inhabitants
gdf["MIG_BAL_RATE"] = (
    gdf["MIG_BAL"]
    /
    gdf["POCET_OBYV"]
) * 1000

# SAVE THE STANDARDIZED DATA

gdf.to_file(
    output_file,
    driver="GeoJSON"
)

print(f"Saved to: {output_file}")

new standardizing code

In [3]:
from pathlib import Path
import geopandas as gpd

def standardize_data(input_path: Path, output_path: Path):
    """
    Reads ORP census data and calculates demographic variables.
    """
    # Load data
    gdf = gpd.read_file(input_path)

    # --- CALCULATIONS ---
    # Area in km²
    gdf["AREA_KM2"] = gdf["SHAPE_Area"] / 1_000_000

    # Population density
    gdf["POP_DENS"] = gdf["POCET_OBYV"] / gdf["AREA_KM2"]

    # Percentages
    gdf["PCT_WOMEN"] = (gdf["ZENY"] / gdf["POCET_OBYV"]) * 100
    gdf["PCT_CHILD"] = (gdf["OBYV_0_14"] / gdf["POCET_OBYV"]) * 100
    gdf["PCT_WORKING"] = (gdf["OBYV_15_64"] / gdf["POCET_OBYV"]) * 100
    gdf["PCT_ELDERLY"] = (gdf["OBYV_65"] / gdf["POCET_OBYV"]) * 100

    # Indices
    gdf["AGEING_INDEX"] = (gdf["OBYV_65"] / gdf["OBYV_0_14"]) * 100
    gdf["DEPENDENCY"] = ((gdf["OBYV_0_14"] + gdf["OBYV_65"]) / gdf["OBYV_15_64"]) * 100

    # Natural Increase
    gdf["NATURAL_INC"] = gdf["NAROZENI"] - gdf["ZEMRELI"]
    gdf["NATURAL_INC_RATE"] = (gdf["NATURAL_INC"] / gdf["POCET_OBYV"]) * 1000

    # Migration Balance
    gdf["MIG_BAL"] = gdf["PRISTEHOVALI"] - gdf["VYSTEHOVALI"]
    gdf["MIG_BAL_RATE"] = (gdf["MIG_BAL"] / gdf["POCET_OBYV"]) * 1000

    # Save the result
    gdf.to_file(output_path, driver="GeoJSON")
    
    return output_path # Returning the path is helpful for the next step in main.py